# RL Peptide Optimizer — Latent Space Navigation

Navigates the HydrAMP latent space using a **Deep Q-Network (DQN)** to find antimicrobial peptides with minimal log₂(MIC) against *E. coli*.

**Design:**
- **State**: 64-dim HydrAMP latent vector of the current peptide
- **Actions**: Candidate mutants generated by MUTANG++ (DecoderLogProbPotential → `compose_mutant_distribution`), represented as latent vectors
- **Reward**: `prev_log2_mic − next_log2_mic` (positive = lower MIC = more potent)
- **Episode**: 20 mutation steps from a fixed seed peptide
- **Exploration**: ε-greedy guided by softmax(MUTANG++ log-potentials)

In [ ]:
import sys, os
# Add project src to path if running from scripts/
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('Project root:', project_root)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch

# Import the RL module (assumes notebook is run from scripts/ or project root)
from rl_peptide_optimizer import (
    run_rl_optimization,
    test_components,
    score_peptides,
    build_apex_predictor,
    ECOLI_INDICES,
)

# ── Configuration ────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

## 1 · Smoke test — verify all components
Runs shape-checked assertions on encoder-decoder, Jacobian SVD, MUTANG++, APEX, and the Q-network.

In [ ]:
test_components(peptide='FLYKWWIRIGRLKL', device=DEVICE)

## 2 · Run RL optimisation

Starting peptides taken from the LEBO paper experiments. Feel free to change `START_PEPTIDE` or `N_EPISODES`.

In [ ]:
# ── Experiment configuration ─────────────────────────────────────────────────
START_PEPTIDE = 'FLYKWWIRIGRLKL'   # "middle-1" from the LEBO experiments
N_EPISODES    = 50
MAX_STEPS     = 20
MAX_CANDIDATES = 30

results = run_rl_optimization(
    start_peptide   = START_PEPTIDE,
    n_episodes      = N_EPISODES,
    max_steps       = MAX_STEPS,
    max_candidates  = MAX_CANDIDATES,
    device          = DEVICE,
    verbose         = True,
    # DQN hyperparameters
    lr              = 1e-3,
    gamma           = 0.99,
    epsilon_start   = 1.0,
    epsilon_end     = 0.05,
    epsilon_decay   = 0.995,
    batch_size      = 32,
    buffer_capacity = 10_000,
    target_update_freq = 50,
)

## 3 · Results summary

In [ ]:
print(f"Start peptide : {results['start_peptide']}")
print(f"Start log2MIC : {results['start_score']:.4f}")
print()
print(f"Best peptide  : {results['best_peptide']}")
print(f"Best log2MIC  : {results['best_score']:.4f}")
print(f"Improvement   : {results['start_score'] - results['best_score']:+.4f} log2 units")
print()
print(f"Episodes run  : {N_EPISODES}")
print(f"Max reward    : {max(results['episode_rewards']):.4f}")
print(f"Mean reward   : {np.mean(results['episode_rewards']):.4f}")

## 4 · Training curves

In [ ]:
episodes = np.arange(1, N_EPISODES + 1)
best_scores   = np.array(results['all_best_scores'])
ep_rewards    = np.array(results['episode_rewards'])
running_best  = np.minimum.accumulate(best_scores)

fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# ── Panel 1: per-episode best log2 MIC ──────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(episodes, best_scores, color='steelblue', lw=1.2, alpha=0.7, label='Episode best')
ax1.plot(episodes, running_best, color='firebrick', lw=2, label='Running best')
ax1.axhline(results['start_score'], color='grey', ls='--', lw=1, label='Start')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Mean log₂(MIC) [E. coli]')
ax1.set_title('Best log₂(MIC) per episode')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# ── Panel 2: episode reward ──────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
window = max(1, N_EPISODES // 10)
smoothed = np.convolve(ep_rewards, np.ones(window)/window, mode='valid')
ax2.plot(episodes, ep_rewards, color='steelblue', lw=1, alpha=0.5, label='Raw')
ax2.plot(episodes[:len(smoothed)], smoothed, color='darkorange', lw=2, label=f'MA-{window}')
ax2.axhline(0, color='grey', ls='--', lw=1)
ax2.set_xlabel('Episode')
ax2.set_ylabel('Total reward')
ax2.set_title('Episode reward')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# ── Panel 3: improvement bar (start vs best) ─────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
labels  = ['Start peptide', 'Best found']
values  = [results['start_score'], results['best_score']]
colors  = ['#5b8db8', '#c0392b']
bars = ax3.bar(labels, values, color=colors, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)
ax3.set_ylabel('Mean log₂(MIC) [E. coli]')
ax3.set_title('Start vs Best peptide')
ax3.grid(axis='y', alpha=0.3)

# ── Panel 4: histogram of all episode-best scores ────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
ax4.hist(best_scores, bins=min(20, N_EPISODES//2 + 1), color='steelblue',
         edgecolor='white', alpha=0.8)
ax4.axvline(results['start_score'], color='grey',     ls='--', lw=1.5, label='Start')
ax4.axvline(results['best_score'],  color='firebrick', ls='-',  lw=1.5, label='Best')
ax4.set_xlabel('Mean log₂(MIC) [E. coli]')
ax4.set_ylabel('Count')
ax4.set_title('Distribution of episode-best scores')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)

fig.suptitle(
    f'DQN Peptide Optimisation  |  start: {START_PEPTIDE}  |  {N_EPISODES} episodes × {MAX_STEPS} steps',
    fontsize=11, y=1.01
)
plt.savefig('rl_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to rl_training_curves.png')

## 5 · Best trajectory inspection

In [ ]:
# Find episode with best final score
best_ep_idx = int(np.argmin(results['all_best_scores']))
best_traj   = results['trajectories'][best_ep_idx]

# Score every peptide in that trajectory
apex = build_apex_predictor(DEVICE)
traj_scores = score_peptides(apex, best_traj)

print(f'Best episode: {best_ep_idx + 1}  (log2MIC={results["all_best_scores"][best_ep_idx]:.4f})')
print(f'Trajectory length: {len(best_traj)} steps\n')
print(f'{"Step":>4}  {"Peptide":<26}  {"log2MIC":>8}')
print('-' * 44)
for i, (seq, sc) in enumerate(zip(best_traj, traj_scores)):
    marker = ' ◄ best' if sc == min(traj_scores) else ''
    print(f'{i:>4}  {seq:<26}  {sc:>8.4f}{marker}')

## 6 · Top-10 peptides found across all episodes

In [ ]:
# Collect all unique peptides from all trajectories
all_seqs = list({seq for traj in results['trajectories'] for seq in traj})
all_scores = score_peptides(apex, all_seqs)

# Sort by score ascending (lower = better)
order = np.argsort(all_scores)
top_k = min(10, len(all_seqs))

print(f'Total unique peptides visited: {len(all_seqs)}')
print(f'\nTop-{top_k} peptides by log₂(MIC) [E.coli mean]:')
print(f'{"Rank":>4}  {"Peptide":<26}  {"log2MIC":>8}')
print('-' * 44)
for rank, idx in enumerate(order[:top_k], 1):
    print(f'{rank:>4}  {all_seqs[idx]:<26}  {all_scores[idx]:>8.4f}')

## 7 · Run additional starting peptides (optional)

Uncomment and run to test other seed peptides from the LEBO paper.

In [ ]:
# OTHER_PEPTIDES = {
#     'jurand-4':      'KYCRRFRWLTFRWL',
#     'jurand-2':      'KFRNRHRWKFKLIFRN',
#     'mammuthusin-3': 'KTLKIIRLLF',
#     'hydrodamin-2':  'RMARNLVRYVQGLKKKKVI',
# }
# 
# all_results = {}
# for name, pep in OTHER_PEPTIDES.items():
#     print(f'\n── {name} ({pep}) ──')
#     r = run_rl_optimization(
#         start_peptide=pep, n_episodes=30, max_steps=20,
#         max_candidates=30, device=DEVICE, verbose=True,
#     )
#     all_results[name] = r
#     print(f'  Best: {r["best_peptide"]}  score={r["best_score"]:.4f}')